# Embeddings: learned vectors for categories

A recipe site tags saved recipes (baking, curry, salad, pasta, breakfast) from their ingredients. In this
notebook you check that an embedding table is a one-hot row times a weight matrix, train the same small recipe
tagger as the lesson, read which ingredients ended up close, and break it with an ingredient id the table has
no row for.

About 10 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime; no GPU, no account.

## 1. Imports

The next cell loads PyTorch. Colab already has it, so nothing is installed.

**Look for:** the PyTorch version. The lesson was run with 2.15. Another version may print slightly different
numbers, but the story stays the same.

In [1]:
import random
import torch
from torch import nn
import torch.nn.functional as F

print("torch version:", torch.__version__)

torch version: 2.15.0


## 2. The saved recipes

This is the lesson's toy data. Each saved recipe is 8 ingredient ids and the collection it was saved into.
Id 0 is the empty slot: a short recipe is padded with 0s. The seed is fixed, so you get exactly the lesson's
recipes: 900 in all, 750 for training and 150 held back.

**Look for:** 45 names (44 ingredients plus the empty slot), and the first recipes: salad, breakfast, baking...

In [2]:
TAGS = ["baking", "curry", "salad", "pasta", "breakfast"]
CORE = {
    "baking":    ["flour", "sugar", "butter", "cinnamon", "nutmeg", "vanilla", "baking powder", "cocoa"],
    "curry":     ["cumin", "turmeric", "cardamom", "coriander", "ginger", "coconut milk", "chili", "lentils"],
    "salad":     ["lettuce", "cucumber", "feta", "olives", "lemon", "red onion", "avocado", "radish"],
    "pasta":     ["spaghetti", "basil", "parmesan", "tomato", "pine nuts", "oregano", "mozzarella", "olive oil"],
    "breakfast": ["oats", "eggs", "bacon", "maple syrup", "yogurt", "berries", "honey", "banana"],
}
STAPLES = ["salt", "garlic", "onion", "black pepper"]
# a few ingredients that also turn up in another collection now and then
CROSS = {"baking": ["eggs", "honey"], "curry": ["tomato", "garlic"], "salad": ["olive oil", "tomato"],
         "pasta": ["garlic", "lemon"], "breakfast": ["cinnamon", "butter"]}
NAMES = ["<empty>"] + [n for t in TAGS for n in CORE[t]] + STAPLES
ID = {n: i for i, n in enumerate(NAMES)}
SLOTS = 8

def make_recipes(n, seed=7):
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        tag = rng.randrange(len(TAGS))
        t = TAGS[tag]
        names = rng.sample(CORE[t], rng.randint(3, 5))
        if rng.random() < 0.6:
            names += rng.sample(STAPLES, rng.randint(1, 2))
        if rng.random() < 0.3:
            c = rng.choice(CROSS[t])
            if c not in names:
                names.append(c)
        ids = [ID[x] for x in names][:SLOTS]
        out.append((ids + [0] * (SLOTS - len(ids)), tag))
    return out

recipes = make_recipes(900)
train, val = recipes[:750], recipes[750:]
print("names incl. empty slot:", len(NAMES), " training recipes:", len(train), " held back:", len(val))
print("one recipe as the model sees it:", train[0])
print("first 12 training recipes:")
for ids, tag in train[:12]:
    print(TAGS[tag], "|", ", ".join(NAMES[i] for i in ids if i))

names incl. empty slot: 45  training recipes: 750  held back: 150
one recipe as the model sees it: ([23, 22, 17, 43, 0, 0, 0, 0], 2)
first 12 training recipes:
salad | avocado, red onion, lettuce, onion
breakfast | oats, banana, maple syrup, salt
baking | sugar, cocoa, vanilla, nutmeg, flour, salt, black pepper, eggs
salad | feta, lemon, lettuce, avocado, salt
curry | turmeric, ginger, coconut milk, cumin, black pepper
pasta | olive oil, pine nuts, tomato, parmesan, garlic, lemon
breakfast | berries, banana, maple syrup, bacon, butter
curry | cardamom, coriander, chili, cumin, garlic
salad | red onion, lemon, olives, avocado, radish, onion
baking | nutmeg, vanilla, cocoa
salad | avocado, red onion, feta, lettuce, olives, black pepper, tomato
curry | coriander, lentils, chili, coconut milk, cumin, onion, salt


## 3. Which collections each ingredient is saved in

The tagger can only learn what the data shows. This cell counts, for four ingredients, how many training
recipes in each collection contain it.

**Look for:** cardamom only in curry (82). Cinnamon in baking and breakfast, never in curry. Salt in every
collection (39, 28, 35, 32, 31), so salt tells you nothing about the collection. Basil only in pasta.

In [3]:
for n in ["cardamom", "cinnamon", "salt", "basil"]:
    counts = {t: 0 for t in TAGS}
    for ids, tag in train:
        if ID[n] in ids:
            counts[TAGS[tag]] += 1
    print("recipes with", n, counts)

recipes with cardamom {'baking': 0, 'curry': 82, 'salad': 0, 'pasta': 0, 'breakfast': 0}
recipes with cinnamon {'baking': 87, 'curry': 0, 'salad': 0, 'pasta': 0, 'breakfast': 30}
recipes with salt {'baking': 39, 'curry': 28, 'salad': 35, 'pasta': 32, 'breakfast': 31}
recipes with basil {'baking': 0, 'curry': 0, 'salad': 0, 'pasta': 61, 'breakfast': 0}


## 4. A lookup is a one-hot row times the table

The site really knows 5,000 ingredients. This cell builds a table of 5,000 rows with 16 numbers each, then gets
basil's row (id 2741) two ways: a one-hot row times the whole weight matrix, and a plain lookup.

**Look for:** the same three first numbers both ways (-0.8403, -0.8617, -1.4625), `same: True`, and the
weight counts: 80,000 for the table, 320,064 for a one-hot row into 64 units, 1,088 for 16 numbers into 64 units.

In [4]:
def same_vector(n_ingredients, dim, basil_id):
    table = nn.Embedding(n_ingredients, dim)
    one_hot = torch.zeros(n_ingredients)
    one_hot[basil_id] = 1.0
    via_matmul = one_hot @ table.weight
    via_lookup = table(torch.tensor(basil_id))
    print("one_hot shape:", tuple(one_hot.shape), " nonzero numbers:", int((one_hot != 0).sum()))
    print("via_matmul first 3:", [round(x, 4) for x in via_matmul[:3].tolist()])
    print("via_lookup first 3:", [round(x, 4) for x in via_lookup[:3].tolist()])
    return torch.equal(via_matmul, via_lookup)

torch.manual_seed(0)
print("same:", same_vector(5000, 16, 2741))
print("weight count of nn.Embedding(5000, 16):", nn.Embedding(5000, 16).weight.numel())
print("weight count of nn.Linear(5000, 64):", sum(p.numel() for p in nn.Linear(5000, 64).parameters()))
print("weight count of nn.Linear(16, 64):", sum(p.numel() for p in nn.Linear(16, 64).parameters()))

one_hot shape: (5000,)  nonzero numbers: 1
via_matmul first 3: [-0.8403, -0.8617, -1.4625]
via_lookup first 3: [-0.8403, -0.8617, -1.4625]
same: True
weight count of nn.Embedding(5000, 16): 80000
weight count of nn.Linear(5000, 64): 320064
weight count of nn.Linear(16, 64): 1088


## 5. The tagger, with an embedding first

`RecipeTagger` looks up every id, averages the real ingredients' vectors (the empty slots are not counted)
into one recipe vector, and a dense layer turns that into 5 collection scores. `padding_idx=0` makes row 0 start
at zeros and never get a gradient. The seed is set again here so the model starts from the lesson's numbers.

**Look for:** weight shape (45, 16) with 720 numbers, 85 in the last layer, row 0 all zeros, and the shapes
(32, 8) -> (32, 8, 16) -> (32, 5): the lookup adds one last axis of size 16.

In [5]:
torch.manual_seed(0)

class RecipeTagger(nn.Module):
    def __init__(self, n_ingredients, dim, n_tags):
        super().__init__()
        self.ingredient_vecs = nn.Embedding(n_ingredients, dim, padding_idx=0)
        self.to_tag = nn.Linear(dim, n_tags)
    def forward(self, ingredient_ids):                    # (batch, 8) ids, 0 = empty slot
        vecs = self.ingredient_vecs(ingredient_ids)       # (batch, 8, dim)
        counts = (ingredient_ids != 0).sum(1, keepdim=True)
        recipe_vec = vecs.sum(1) / counts                 # mean of the real ingredients
        return self.to_tag(recipe_vec)

def tensors(rs):
    return torch.tensor([r[0] for r in rs]), torch.tensor([r[1] for r in rs])
Xtr, ytr = tensors(train); Xva, yva = tensors(val)

model = RecipeTagger(len(NAMES), 16, len(TAGS))
print("ingredients incl. empty slot:", len(NAMES))
print("embedding weight shape:", tuple(model.ingredient_vecs.weight.shape),
      "params:", model.ingredient_vecs.weight.numel())
print("to_tag params:", sum(p.numel() for p in model.to_tag.parameters()))
print("empty-slot row before training, all zeros:", bool((model.ingredient_vecs.weight[0] == 0).all()))
b = Xtr[:32]
print("batch ids shape:", tuple(b.shape), "-> vecs shape:", tuple(model.ingredient_vecs(b).shape),
      "-> logits shape:", tuple(model(b).shape))

ingredients incl. empty slot: 45
embedding weight shape: (45, 16) params: 720
to_tag params: 85
empty-slot row before training, all zeros: True
batch ids shape: (32, 8) -> vecs shape: (32, 8, 16) -> logits shape: (32, 5)


## 6. Nearest ingredients before training

`neighbours` compares one ingredient's row with every other row by cosine similarity (1 = same direction,
0 = unrelated, below 0 = pointing apart) and returns the top 3. The rows are still random.

**Look for:** neighbours that make no sense yet, like cardamom next to basil and cocoa.

In [6]:
def neighbours(name, k=3):
    W = model.ingredient_vecs.weight.detach()
    sims = F.cosine_similarity(W[ID[name]].unsqueeze(0), W, dim=1)
    sims[0] = -2; sims[ID[name]] = -2
    top = sims.topk(k)
    return [(NAMES[i], round(s, 2)) for s, i in zip(top.values.tolist(), top.indices.tolist())]

WATCH = ["cardamom", "basil", "salt", "cinnamon"]
start = model.ingredient_vecs.weight.detach().clone()
for n in WATCH:
    print("BEFORE", n, neighbours(n))

BEFORE cardamom [('turmeric', 0.39), ('basil', 0.34), ('cocoa', 0.32)]
BEFORE basil [('cucumber', 0.56), ('butter', 0.51), ('olives', 0.5)]
BEFORE salt [('cocoa', 0.53), ('basil', 0.41), ('red onion', 0.38)]
BEFORE cinnamon [('bacon', 0.56), ('red onion', 0.48), ('cumin', 0.43)]


## 7. Train

30 passes over the 750 training recipes, batches of 32, Adam. The embedding rows are trained like any other
weights. This takes a few seconds.

**Look for:** the training loss falling at every printed epoch, from about 1.215 to about 0.001, and validation
accuracy reaching 1.000. This toy set is easy.

In [7]:
opt = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(1, 31):
    perm = torch.randperm(len(Xtr))
    total = 0.0
    for s in range(0, len(Xtr), 32):
        idx = perm[s:s + 32]
        opt.zero_grad()
        loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
        loss.backward()
        opt.step()
        total += loss.item() * len(idx)
    with torch.no_grad():
        acc = (model(Xva).argmax(1) == yva).float().mean().item()
    if epoch in (1, 2, 3, 5, 10, 20, 30):
        print(f"epoch {epoch:2d}  train loss {total / len(Xtr):.3f}  val accuracy {acc:.3f}")

epoch  1  train loss 1.215  val accuracy 0.967
epoch  2  train loss 0.494  val accuracy 1.000
epoch  3  train loss 0.127  val accuracy 1.000
epoch  5  train loss 0.024  val accuracy 1.000
epoch 10  train loss 0.006  val accuracy 1.000
epoch 20  train loss 0.002  val accuracy 1.000


epoch 30  train loss 0.001  val accuracy 1.000


## 8. Nearest ingredients after training

Same four ingredients, now with trained rows.

**Look for:** row 0 still all zeros. Cardamom now next to curry ingredients (turmeric, coriander, chili). Basil
next to pasta ingredients. Cinnamon next to baking ingredients, not cardamom. Salt still next to basil and
cocoa: those neighbours mean nothing (next cell).

In [8]:
print("empty-slot row after training, still all zeros:", bool((model.ingredient_vecs.weight[0] == 0).all()))
for n in WATCH:
    print("AFTER", n, neighbours(n))

empty-slot row after training, still all zeros: True
AFTER cardamom [('turmeric', 0.78), ('coriander', 0.74), ('chili', 0.57)]
AFTER basil [('pine nuts', 0.67), ('olive oil', 0.62), ('parmesan', 0.6)]
AFTER salt [('basil', 0.36), ('cocoa', 0.32), ('onion', 0.28)]
AFTER cinnamon [('sugar', 0.48), ('cocoa', 0.4), ('baking powder', 0.4)]


## 9. How far each row moved, and a few cosines

Every row started as random numbers; only the gradient moves it. This cell prints how far each watched row
moved during training, then the cosine of a few pairs.

**Look for:** salt moved about 0.79, the others about 3. Cardamom and cumin (both curry) at 0.46, cardamom and
cinnamon at -0.2: in these recipes they never share a collection, so they point slightly apart.

In [9]:
W = model.ingredient_vecs.weight.detach()
for n in WATCH:
    print("moved", n, "length at start", round(start[ID[n]].norm().item(), 2), "distance moved", round((W[ID[n]] - start[ID[n]]).norm().item(), 2))
for a, c in [("cardamom", "cumin"), ("cardamom", "cinnamon"), ("cinnamon", "nutmeg"), ("salt", "cumin"), ("basil", "oregano"), ("basil", "cardamom")]:
    print("cos", a, c, round(F.cosine_similarity(W[ID[a]], W[ID[c]], dim=0).item(), 2))

moved cardamom length at start 3.89 distance moved 3.05
moved basil length at start 4.02 distance moved 3.38
moved salt length at start 4.16 distance moved 0.79
moved cinnamon length at start 4.43 distance moved 3.08
cos cardamom cumin 0.46
cos cardamom cinnamon -0.2
cos cinnamon nutmeg 0.36
cos salt cumin -0.03
cos basil oregano 0.32
cos basil cardamom 0.1


## 10. Broken on purpose: a new ingredient

A week after training, the site adds gochujang and gives it the next free id, 45. The trained table has rows 0
to 44 only. The call is wrapped in `try/except` so the notebook keeps going.

**Look for:** `table rows: 45  new ingredient id: 45`, then `IndexError: index out of range in self`.

In [10]:
names_now = NAMES + ["gochujang"]          # the site's ingredient list after the new ingredient was added
new_id = len(names_now) - 1
print("table rows:", model.ingredient_vecs.num_embeddings, " new ingredient id:", new_id)
saved = [ID["chili"], ID["garlic"], new_id, ID["ginger"], 0, 0, 0, 0]
print("saved recipe ids:", saved)
try:
    model(torch.tensor([saved]))
except IndexError as e:
    print("IndexError:", e)

table rows: 45  new ingredient id: 45
saved recipe ids: [15, 42, 45, 13, 0, 0, 0, 0]
IndexError: index out of range in self


# Now change it

Three small experiments. Each cell has one line marked `# <- change this`. Edit it, run the cell, and compare.

### Experiment 1: another ingredient

Lemon is in salad recipes and sometimes in pasta, so its neighbours mix both. Change `name` to `"oats"`, then to
`"garlic"`. Oats appear only in breakfast recipes: their nearest rows should be breakfast ingredients, and the
row should move far (about 3, like cardamom). Garlic is a staple in every collection that is also extra common in
curry and pasta: it should move about half as far, with curry and pasta ingredients as neighbours.

In [11]:
name = "lemon"                                    # <- change this
print("nearest to", name, neighbours(name))
print("distance moved:", round((W[ID[name]] - start[ID[name]]).norm().item(), 2))

nearest to lemon [('avocado', 0.57), ('mozzarella', 0.56), ('parmesan', 0.53)]
distance moved: 1.97


### Experiment 2: fewer numbers per ingredient

The table width is your choice. This cell trains a fresh tagger with `DIM` numbers per ingredient. Change `DIM`
from 16 to 2. The table shrinks from 720 to 90 numbers. Watch the validation accuracy and cardamom's neighbours.
This toy set is so easy that even 2 numbers still tag every held-back recipe right, and cardamom's neighbours
are still curry ingredients (other ones). The width you need depends on how hard the task is.

In [12]:
DIM = 16                                          # <- change this
torch.manual_seed(0)
m2 = RecipeTagger(len(NAMES), DIM, len(TAGS))
opt2 = torch.optim.Adam(m2.parameters(), lr=0.01)
for epoch in range(1, 31):
    perm = torch.randperm(len(Xtr))
    for s in range(0, len(Xtr), 32):
        idx = perm[s:s + 32]
        opt2.zero_grad()
        F.cross_entropy(m2(Xtr[idx]), ytr[idx]).backward()
        opt2.step()
with torch.no_grad():
    acc2 = (m2(Xva).argmax(1) == yva).float().mean().item()
W2 = m2.ingredient_vecs.weight.detach()
sims = F.cosine_similarity(W2[ID["cardamom"]].unsqueeze(0), W2, dim=1)
sims[0] = -2; sims[ID["cardamom"]] = -2
print("table numbers:", W2.numel(), " val accuracy:", round(acc2, 3))
print("nearest to cardamom:", [NAMES[i] for i in sims.topk(3).indices.tolist()])

table numbers: 720  val accuracy: 1.0
nearest to cardamom: ['turmeric', 'coriander', 'chili']


### Experiment 3: a planned place for unseen ids

The lesson's fix is a reserved "unknown" row. This toy table has none, so the cell sends any id the table has no
row for to `UNKNOWN`. It starts at 0, the empty slot: gochujang is then simply ignored, and the recipe is tagged
from chili, garlic and ginger. Change `UNKNOWN` to 44: there is no crash either, but gochujang now silently uses
row 44, which is black pepper. That is why clamping ids is not a fix.

In [13]:
UNKNOWN = 0                                       # <- change this
rows = model.ingredient_vecs.num_embeddings
safe = [i if i < rows else UNKNOWN for i in saved]
with torch.no_grad():
    scores = model(torch.tensor([safe]))
print("ids sent:", safe, " row", UNKNOWN, "is", NAMES[UNKNOWN])
print("tag:", TAGS[scores.argmax(1).item()])

ids sent: [15, 42, 0, 13, 0, 0, 0, 0]  row 0 is <empty>
tag: curry


## What you should see

- Step 5 prints `embedding weight shape: (45, 16)` with 720 learned numbers.
- Step 5 prints batch ids (32, 8) -> vecs (32, 8, 16) -> logits (32, 5).
- Step 7: the training loss falls at every printed epoch, from about 1.215 to about 0.001.
- Step 8: row 0, the empty slot, is still all zeros after training.
- Step 8: after training, cardamom's nearest rows are curry ingredients (turmeric, coriander, chili).
- Step 9: salt's row moved far less than cardamom's (0.79 against 3.05).